# Model Selection and Error Analysis for `difficulty_results_model_summary.csv`

This notebook does two things:
1. Uses `difficulty_results_model_summary.csv` to identify the best model configuration from CV summary metrics.
2. Rebuilds that best model on the original dataset split so we can compute per-rating accuracy and confusion matrix (not stored in the CSV).


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')

RESULTS_CSV = 'difficulty_results_model_summary.csv'
results = pd.read_csv(RESULTS_CSV)
results.head()


In [ ]:
# Coerce metric columns to numeric for reliable sorting/filtering
metric_cols = [
    'accuracy', 'macro_f1', 'mae',
    'mean_accuracy', 'mean_macro_f1', 'mean_mae',
    'std_accuracy', 'std_macro_f1', 'std_mae', 'fold', 'k'
]
for c in metric_cols:
    if c in results.columns:
        results[c] = pd.to_numeric(results[c], errors='coerce')

print('Rows:', len(results))
print('Result types:', sorted(results['result_type'].dropna().unique().tolist()))
results.groupby('result_type').size().rename('n_rows').to_frame()


## Prepare model summary table

This notebook assumes the input file is already the difficulty model-summary table (CV summary + optional test row).


In [ ]:
if 'result_type' in results.columns:
    model_summary_only = results[results['result_type'].isin(['cv_summary', 'test'])].copy()
else:
    model_summary_only = results.copy()

print(f'Model summary rows: {len(model_summary_only)}')
display(model_summary_only.head())


In [ ]:
# Find best config from CV summary rows
cv_summary = model_summary_only[model_summary_only['result_type'] == 'cv_summary'].copy()
cv_summary = cv_summary.sort_values(
    ['mean_accuracy', 'mean_macro_f1', 'mean_mae'],
    ascending=[False, False, True]
).reset_index(drop=True)

if cv_summary.empty:
    raise ValueError('No cv_summary rows found in model summary table')

best_cv = cv_summary.iloc[0]
best_model_name = str(best_cv['model'])
best_k = int(best_cv['k'])

print('Best model from CV summary:')
display(best_cv.to_frame('value'))

print('Top 10 CV summary rows:')
display(cv_summary[['model', 'k', 'mean_accuracy', 'mean_macro_f1', 'mean_mae']].head(10))


In [ ]:
# Check reported test metric row in the model summary table (optional; may be absent by runner config)
if 'result_type' in model_summary_only.columns:
    test_rows = model_summary_only[model_summary_only['result_type'] == 'test'].copy()
else:
    test_rows = pd.DataFrame()

test_display_cols = ['model', 'k', 'accuracy', 'macro_f1', 'mae']
available_cols = [c for c in test_display_cols if c in test_rows.columns]

if test_rows.empty:
    print('No test row found in model summary CSV (expected when runner include_test_row=False).')
elif len(available_cols) < len(test_display_cols):
    print('Test row exists but some metric columns are missing; showing available columns only.')
    display(test_rows[available_cols])
else:
    display(test_rows[test_display_cols])
    row = test_rows.iloc[0]
    print(f"CSV test row -> model={row['model']}, k={int(row['k'])}, accuracy={row['accuracy']:.4f}, macro_f1={row['macro_f1']:.4f}, mae={row['mae']:.4f}")


## Rebuild best model for per-rating analysis

`difficulty_results_model_summary.csv` stores aggregate metrics only; it does not store `y_true`/`y_pred`.
The next cells reconstruct the same train/test split and best configuration, then compute:
- per-rating accuracy when the true label is 1..5
- a classification report
- a confusion matrix


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rating_data import ReviewDatasetManager
from rating_features import FeaturePipeline
from rating_models import ModelRegistry
from rating_runner_core import ModelEvaluator


In [ ]:
# Recreate the same split and feature pipeline used by difficulty_runner default config
INPUT_CSV = 'rmp_all_schools_reviews_small.csv'
TARGET_COL = 'difficultyRating'
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_ROWS = None

dataset_manager = ReviewDatasetManager(target_col=TARGET_COL)
feature_pipeline = FeaturePipeline(random_state=RANDOM_STATE)
spec_map = {spec.name: spec for spec in ModelRegistry(random_state=RANDOM_STATE).get_specs()}
best_spec = spec_map[best_model_name]

df = dataset_manager.load_and_validate(INPUT_CSV, MAX_ROWS)
tag_cols = dataset_manager.get_tag_columns(df)
df_train, df_test = dataset_manager.split_train_test(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)

train_comments = df_train['comment_clean'].astype(str).tolist()
test_comments = df_test['comment_clean'].astype(str).tolist()
y_train = df_train[TARGET_COL].to_numpy()
y_test = df_test[TARGET_COL].to_numpy()

train_tags = feature_pipeline.tag_matrix(df_train, tag_cols)
test_tags = feature_pipeline.tag_matrix(df_test, tag_cols)

vectorizer, nmf, x_train = feature_pipeline.fit_transform(train_comments, train_tags, k=best_k)
x_test = feature_pipeline.transform(test_comments, test_tags, vectorizer, nmf)

y_pred = ModelEvaluator.fit_predict_model(best_spec, x_train, y_train, x_test, min_label=1, max_label=5)

test_accuracy = accuracy_score(y_test, y_pred)
test_macro_f1 = f1_score(y_test, y_pred, average='macro')
test_mae = mean_absolute_error(y_test, y_pred)

print(f'Rebuilt best model: {best_model_name} (k={best_k})')
print(f'Holdout accuracy: {test_accuracy:.4f}')
print(f'Holdout macro F1: {test_macro_f1:.4f}')
print(f'Holdout MAE: {test_mae:.4f}')


In [ ]:
labels = [1, 2, 3, 4, 5]
cm = confusion_matrix(y_test, y_pred, labels=labels)

row_totals = cm.sum(axis=1)
diag = np.diag(cm)
per_rating_accuracy = np.divide(diag, row_totals, out=np.zeros_like(diag, dtype=float), where=row_totals != 0)

per_rating_df = pd.DataFrame({
    'true_rating': labels,
    'support': row_totals,
    'correct_predictions': diag,
    'accuracy_when_true': per_rating_accuracy,
})

print('Per-rating accuracy (conditioned on true class):')
display(per_rating_df)

print('Classification report:')
print(classification_report(y_test, y_pred, labels=labels, digits=4))

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix: {best_model_name} (k={best_k})')
plt.xlabel('Predicted rating')
plt.ylabel('True rating')
plt.tight_layout()
plt.show()

cm_norm = cm / np.clip(row_totals[:, None], 1, None)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', xticklabels=labels, yticklabels=labels)
plt.title(f'Row-normalized Confusion Matrix: {best_model_name} (k={best_k})')
plt.xlabel('Predicted rating')
plt.ylabel('True rating')
plt.tight_layout()
plt.show()


## Interpretable language and topic takeaways

The goal of this section is not to claim causality. It gives a small set of descriptive patterns from the training split only:
- which frequent rating tags tend to appear with higher or lower difficulty ratings
- which NMF topics have higher or lower average observed ratings
- a short plain-English summary of what the model seems to be picking up


In [ ]:
# Tag-level descriptive patterns
min_tag_support = max(75, int(0.02 * len(df_train)))
tag_rows = []
for col in tag_cols:
    support = int(df_train[col].sum())
    if support < min_tag_support:
        continue
    present_mean = df_train.loc[df_train[col] == 1, TARGET_COL].mean()
    absent_mean = df_train.loc[df_train[col] == 0, TARGET_COL].mean()
    tag_rows.append({
        'tag': col.replace('tag_', ''),
        'support': support,
        'share_of_train': support / len(df_train),
        'mean_rating_when_present': present_mean,
        'rating_lift_vs_absent': present_mean - absent_mean,
    })

tag_summary = pd.DataFrame(tag_rows).sort_values(
    ['rating_lift_vs_absent', 'support'], ascending=[False, False]
).reset_index(drop=True)

top_positive_tags = tag_summary.head(5).copy()
top_negative_tags = tag_summary.tail(5).sort_values('rating_lift_vs_absent').copy()

print('Frequent tags associated with higher observed difficulty ratings:')
display(top_positive_tags[['tag', 'support', 'mean_rating_when_present', 'rating_lift_vs_absent']])
print('Frequent tags associated with lower observed difficulty ratings:')
display(top_negative_tags[['tag', 'support', 'mean_rating_when_present', 'rating_lift_vs_absent']])

# Topic-level descriptive patterns from the fitted NMF on the training split
x_train_tfidf = vectorizer.transform(train_comments)
W_train = nmf.transform(x_train_tfidf)
H = nmf.components_
feature_names = np.array(vectorizer.get_feature_names_out())
dominant_topic = W_train.argmax(axis=1)
dominant_strength = W_train.max(axis=1)

topic_rows = []
for topic_id in range(nmf.n_components):
    mask = dominant_topic == topic_id
    support = int(mask.sum())
    top_words = feature_names[np.argsort(H[topic_id])[::-1][:8]].tolist()
    topic_rows.append({
        'topic': topic_id,
        'support': support,
        'mean_rating': float(y_train[mask].mean()) if support else np.nan,
        'top_words': ', '.join(top_words),
        'example_strength_mean': float(dominant_strength[mask].mean()) if support else np.nan,
    })

topic_summary = pd.DataFrame(topic_rows).sort_values(['mean_rating', 'support'], ascending=[False, False]).reset_index(drop=True)
print('Topics ranked by average observed difficulty rating:')
display(topic_summary[['topic', 'support', 'mean_rating', 'top_words']])


## Small set of takeaways

After running the previous cell, summarize the strongest patterns in plain language here. Keep them descriptive, not causal. For example:

- Among the frequent tags, the tags near the top of the positive table are associated with higher average difficulty ratings, while the tags near the top of the negative table are associated with lower average difficulty ratings.
- The highest-rated NMF topics tend to contain words that signal harder workload, tougher grading, or more challenging assessments, while the lowest-rated topics tend to contain words that describe easier pacing, lighter workload, or more accessible expectations.
- These are correlations from the training split, not causal claims about what makes a class difficult.
